# HRM Excel Sheet Mapping Ingestion - All 14 Tables + snapshot_date_ts

Notebook này chạy đủ **14 bảng** trong `SHEET_MAPPING`:

1. `hr_training_instructor_list`
2. `hr_training_learner_list`
3. `hr_training_online_learning_list`
4. `hr_training_employee_certificate_list`
5. `hr_training_budget_summary_by_type`
6. `hr_training_budget_expense_detail`
7. `hr_training_budget_summary_by_quarter`
8. `hr_training_trainee_list`
9. `hr_training_trainee_conversion_list`
10. `hr_training_trainee_resignation_list`
11. `hr_training_fresher_list`
12. `hr_training_fresher_conversion_list`
13. `hr_training_fresher_resignation_list`
14. `hr_training_class_list`

Luồng chạy:

1. Load `.env.minio`
2. Download/test file Excel
3. Test cấu trúc từng bảng
4. Xử lý toàn bộ 14 bảng
5. Ghi Parquet từng bảng
6. Upload từng bảng lên raw bucket
7. Execute SQL create table từng bảng ở bước cuối

Riêng sheet `06.Ngân sách` được tách theo range:

- `A1:F12` → `hr_training_budget_summary_by_type`
- `G1:R` → `hr_training_budget_expense_detail`
- `B22:F26` → `hr_training_budget_summary_by_quarter`


Bản này bổ sung thêm metadata từ tên file:

- `snapshot_date` dạng `string/varchar`, ví dụ `2026-06-18`
- `snapshot_date_ts` dạng `bigint`, Unix timestamp tại 00:00:00 UTC của `snapshot_date`


## 1. Import, env, paths

In [1]:
import os
import re
import json
import time
import tempfile
import urllib3
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from datetime import datetime, date, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv, dotenv_values
from minio import Minio
from minio.error import InvalidResponseError, S3Error

load_dotenv(".env.minio", override=True)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Nếu notebook đang chạy ở:
# /prefecthq-external-ingestion-mino/ingestions/data_tools
# thì resources nằm ở:
# /prefecthq-external-ingestion-mino/ingestions/resources
RESOURCES_DIR = Path.cwd().parent / "resources"

MAPPING_DIR = RESOURCES_DIR / "mappings"
PARQUET_SCHEMA_DIR = RESOURCES_DIR / "parquet_schema"
HIVE_SQL_DIR = RESOURCES_DIR / "hive_sql"

DEFAULT_DOMAIN = "hr_raw"

# SQL:
# true  = chỉ print SQL, không execute
# false = execute SQL thật ở bước cuối
os.environ.setdefault("DRY_RUN_SQL", "true")

print("Current working dir:", Path.cwd())
print(".env.minio exists:", Path(".env.minio").exists())
print("Loaded dotenv keys:", list(dotenv_values(".env.minio").keys()))
print("RESOURCES_DIR:", RESOURCES_DIR)
print("RESOURCES_DIR exists:", RESOURCES_DIR.exists())

print("MINIO endpoint:", os.getenv("MINIO_HRM_ENDPOINT") or os.getenv("MINIO_ENDPOINT"))
print("MINIO bucket:", os.getenv("MINIO_HRM_BUCKET") or os.getenv("MINIO_BUCKET"))
print("Access key exists:", bool(os.getenv("MINIO_HRM_ACCESS_KEY") or os.getenv("MINIO_ACCESS_KEY")))
print("Secret key exists:", bool(os.getenv("MINIO_HRM_SECRET_KEY") or os.getenv("MINIO_SECRET_KEY")))
print("DRY_RUN_SQL:", os.getenv("DRY_RUN_SQL"))


Current working dir: e:\GitLab\crawlers\prefecthq-external-ingestion\ingestions\data_tools
.env.minio exists: True
Loaded dotenv keys: ['AIRFLOW_UID', 'AMBARI_USERNAME', 'AMBARI_PASSWORD', 'SURVICATE_API_KEY', 'FRESHDESK_API_KEY', 'FRESHWORKS_API_KEY', 'FRESHWORKS_API_COOKIE', 'HDFS_CLIENT', 'NAMENODES_HOST', 'NAMENODES_PORT', 'NAMENODES_SCHEME', 'CRAWLER_HTTP_PROXY', 'CRAWLER_HTTPS_PROXY', 'CRAWLER_NO_PROXY', 'MINIO_NOC_ENDPOINT', 'MINIO_NOC_ACCESS_KEY', 'MINIO_NOC_SECRET_KEY', 'MINIO_NOC_BUCKET', 'MINIO_JIRA_ENDPOINT', 'MINIO_JIRA_ACCESS_KEY', 'MINIO_JIRA_SECRET_KEY', 'MINIO_JIRA_BUCKET', 'MINIO_HRM_ENDPOINT', 'MINIO_HRM_ACCESS_KEY', 'MINIO_HRM_SECRET_KEY', 'MINIO_HRM_BUCKET', 'NOCODB_TABLE_ID', 'NOCODB_TOKEN', 'NOCODB_URL', 'PREFECT_SERVER_ANALYTICS_ENABLED', 'PREFECT_API_ANALYTICS_ENABLED', 'MINIO_ACCESS_KEY', 'MINIO_SECRET_KEY', 'MINIO_ENDPOINT', 'TRINO_HOST', 'TRINO_PORT', 'TRINO_USERNAME', 'TRINO_PASSWORD', 'TRINO_CATALOG', 'TRINO_HTTP_SCHEME', 'DRY_RUN_SQL', 'TRINO_SSL_VERIFY']

## 2. Resource config

In [2]:
from typing import Any

SHEET_MAPPING: dict[str, dict[str, Any]] = {
    "hr_training_class_list": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "01. DS Lớp học", "header_row": 3, "drop_rows": 0, "date_columns": ["updated_date", "start_date", "end_date"]},
    "hr_training_instructor_list": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "02. DS Giảng viên", "header_row": 2, "drop_rows": 0, "date_columns": ["start_date", "end_date"]},
    "hr_training_learner_list": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "03. DS Học viên", "header_row": 2, "drop_rows": 0, "date_columns": ["start_date", "end_date"]},
    "hr_training_online_learning_list": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "04. DS học trực tuyến", "header_row": 2, "drop_rows": 0, "date_columns": []},
    "hr_training_employee_certificate_list": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "05.Chứng chỉ", "header_row": 3, "drop_rows": 0, "date_columns": ["certificate_issue_date", "certificate_expiry_date"]},

    "hr_training_budget_summary_by_type": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "06.Ngân sách", "table_range": "A1:F12", "drop_rows": 0, "date_columns": []},
    "hr_training_budget_expense_detail": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "06.Ngân sách", "table_range": "G1:R", "drop_rows": 0, "date_columns": ["settlement_date"]},
    "hr_training_budget_summary_by_quarter": {"domain": "hr_raw", "object_key_pattern": "L&D/L&D_Master Data_*.xlsx", "sheet_name": "06.Ngân sách", "table_range": "B22:F26", "drop_rows": 0, "date_columns": []},

    "hr_training_trainee_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS sinh viên", "header_row": 1, "drop_rows": 0, "date_columns": ["ojt_start_date", "ojt_end_date"]},
    "hr_training_trainee_conversion_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS sinh viên chuyển diện", "header_row": 1, "drop_rows": 0, "date_columns": ["conversion_date", "expected_full_time_date"]},
    "hr_training_trainee_resignation_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS sinh viên nghỉ việc", "header_row": 0, "drop_rows": 0, "date_columns": ["resigned_date", "expected_full_time_date"]},

    "hr_training_fresher_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS Fresher", "header_row": 1, "drop_rows": 0, "date_columns": ["onboard_date", "ojt_start_date", "ojt_end_date"]},
    "hr_training_fresher_conversion_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS Fresher chuyển diện", "header_row": 1, "drop_rows": 0, "date_columns": ["onboard_date", "conversion_date"]},
    "hr_training_fresher_resignation_list": {"domain": "hr_raw", "object_key_pattern": "L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx", "sheet_name": "DS Fresher nghỉ việc", "header_row": 1, "drop_rows": 0, "date_columns": ["onboard_date", "resigned_date", "ojt_start_date", "ojt_end_date"]},
}

print("Configured resources:", len(SHEET_MAPPING))
for resource_name, config in SHEET_MAPPING.items():
    print("-", resource_name, "|", config["sheet_name"], "|", config.get("table_range", "full sheet"))


Configured resources: 14
- hr_training_class_list | 01. DS Lớp học | full sheet
- hr_training_instructor_list | 02. DS Giảng viên | full sheet
- hr_training_learner_list | 03. DS Học viên | full sheet
- hr_training_online_learning_list | 04. DS học trực tuyến | full sheet
- hr_training_employee_certificate_list | 05.Chứng chỉ | full sheet
- hr_training_budget_summary_by_type | 06.Ngân sách | A1:F12
- hr_training_budget_expense_detail | 06.Ngân sách | G1:R
- hr_training_budget_summary_by_quarter | 06.Ngân sách | B22:F26
- hr_training_trainee_list | DS sinh viên | full sheet
- hr_training_trainee_conversion_list | DS sinh viên chuyển diện | full sheet
- hr_training_trainee_resignation_list | DS sinh viên nghỉ việc | full sheet
- hr_training_fresher_list | DS Fresher | full sheet
- hr_training_fresher_conversion_list | DS Fresher chuyển diện | full sheet
- hr_training_fresher_resignation_list | DS Fresher nghỉ việc | full sheet


## Verify all 14 resources


In [3]:
EXPECTED_RESOURCE_COUNT = 14

print("Total resources:", len(SHEET_MAPPING))
for idx, resource_name in enumerate(SHEET_MAPPING.keys(), start=1):
    print(f"{idx:02d}. {resource_name}")

assert len(SHEET_MAPPING) == EXPECTED_RESOURCE_COUNT, (
    f"Expected {EXPECTED_RESOURCE_COUNT} resources, got {len(SHEET_MAPPING)}"
)


Total resources: 14
01. hr_training_class_list
02. hr_training_instructor_list
03. hr_training_learner_list
04. hr_training_online_learning_list
05. hr_training_employee_certificate_list
06. hr_training_budget_summary_by_type
07. hr_training_budget_expense_detail
08. hr_training_budget_summary_by_quarter
09. hr_training_trainee_list
10. hr_training_trainee_conversion_list
11. hr_training_trainee_resignation_list
12. hr_training_fresher_list
13. hr_training_fresher_conversion_list
14. hr_training_fresher_resignation_list


## 3. MinIO/S3 helpers

In [4]:
def env_get(*keys: str, default: str | None = None) -> str | None:
    for key in keys:
        value = os.getenv(key)
        if value:
            return value
    return default


def get_hrm_bucket() -> str:
    return env_get("MINIO_HRM_BUCKET", "MINIO_BUCKET", default="hrm-data")


def get_raw_bucket() -> str:
    return env_get("MINIO_RAW_BUCKET", "MINIO_BUCKET", default="vcs-raw")


def build_minio_client(endpoint: str, access_key: str, secret_key: str) -> Minio:
    if not endpoint:
        raise ValueError("Missing MinIO endpoint")

    parsed = urlparse(endpoint)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"Invalid MinIO endpoint: {endpoint}")

    # Internal/self-signed cert.
    http_client = urllib3.PoolManager(cert_reqs="CERT_NONE")

    return Minio(
        endpoint=parsed.netloc,
        access_key=access_key,
        secret_key=secret_key,
        secure=parsed.scheme == "https",
        http_client=http_client,
    )


def get_hrm_client() -> Minio:
    endpoint = env_get("MINIO_HRM_ENDPOINT", "MINIO_ENDPOINT")
    access_key = env_get("MINIO_HRM_ACCESS_KEY", "MINIO_ACCESS_KEY")
    secret_key = env_get("MINIO_HRM_SECRET_KEY", "MINIO_SECRET_KEY")

    if not endpoint:
        raise ValueError("Missing MINIO_HRM_ENDPOINT or MINIO_ENDPOINT")
    if not access_key or not secret_key:
        raise ValueError("Missing MinIO access key or secret key")

    return build_minio_client(endpoint, access_key, secret_key)


def get_raw_client() -> Minio:
    endpoint = env_get("MINIO_RAW_ENDPOINT", "MINIO_HRM_ENDPOINT", "MINIO_ENDPOINT")
    access_key = env_get("MINIO_RAW_ACCESS_KEY", "MINIO_HRM_ACCESS_KEY", "MINIO_ACCESS_KEY")
    secret_key = env_get("MINIO_RAW_SECRET_KEY", "MINIO_HRM_SECRET_KEY", "MINIO_SECRET_KEY")

    if not endpoint:
        raise ValueError("Missing raw MinIO endpoint")
    if not access_key or not secret_key:
        raise ValueError("Missing raw MinIO access key or secret key")

    return build_minio_client(endpoint, access_key, secret_key)


def extract_snapshot_date(filename: str) -> date | None:
    """
    Extract snapshot date from filename suffix: _ddmmyyyy.xlsx -> YYYY-MM-DD.
    Example: L&D_Master Data_18062026.xlsx -> date(2026, 6, 18)
    """
    m = re.search(r"_(\d{8})(?:\.[^.]+)?$", Path(filename).name)
    if not m:
        return None

    return datetime.strptime(m.group(1), "%d%m%Y").date()




def date_to_unix_seconds(value: date | None) -> int | None:
    """Convert date to Unix timestamp seconds at 00:00:00 UTC."""
    if value is None:
        return None

    return int(datetime.combine(value, datetime.min.time(), tzinfo=timezone.utc).timestamp())

def resolve_latest_object_key_by_pattern(
    minio_client: Minio,
    bucket_name: str,
    object_key_pattern: str,
) -> tuple[str, date]:
    """
    Resolve latest object by pattern with date suffix.

    Examples:
      L&D/L&D_Master Data_*.xlsx
      L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
    """
    folder_prefix = object_key_pattern.rsplit("/", 1)[0] + "/"

    # Escape special chars like [VCS], then replace escaped * by date regex.
    regex = "^" + re.escape(object_key_pattern).replace("\\*", r"\d{8}") + "$"

    candidates: list[tuple[date, str]] = []

    objects = minio_client.list_objects(
        bucket_name,
        prefix=folder_prefix,
        recursive=True,
    )

    for obj in objects:
        key = obj.object_name

        if re.match(regex, key):
            snapshot_date = extract_snapshot_date(key)
            if snapshot_date:
                candidates.append((snapshot_date, key))

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy file theo pattern: {object_key_pattern}"
        )

    candidates.sort(key=lambda x: x[0], reverse=True)

    latest_snapshot_date, latest_object_key = candidates[0]

    return latest_object_key, latest_snapshot_date


def download_hrm_object(
    object_key: str | None = None,
    object_key_pattern: str | None = None,
    local_dir: str | None = None,
) -> tuple[Path, str, date | None]:
    """
    Download HRM Excel object.

    Returns:
      local_path, resolved_object_key, snapshot_date
    """
    bucket = get_hrm_bucket()
    client = get_hrm_client()

    if object_key_pattern:
        resolved_object_key, snapshot_date = resolve_latest_object_key_by_pattern(
            minio_client=client,
            bucket_name=bucket,
            object_key_pattern=object_key_pattern,
        )
    elif object_key:
        resolved_object_key = object_key
        snapshot_date = extract_snapshot_date(resolved_object_key)
    else:
        raise ValueError("Missing object_key or object_key_pattern")

    target_dir = Path(local_dir or tempfile.mkdtemp(prefix="hrm_ingestion_"))
    target_dir.mkdir(parents=True, exist_ok=True)

    local_path = target_dir / Path(resolved_object_key).name

    client.fget_object(
        bucket_name=bucket,
        object_name=resolved_object_key,
        file_path=str(local_path),
    )

    return local_path, resolved_object_key, snapshot_date


from openpyxl.utils import column_index_from_string


def parse_excel_range(table_range: str):
    """
    Supports:
      A1:F12
      G1:R
      B22:F26
    """
    match = re.match(r"^([A-Z]+)(\d+):([A-Z]+)(\d*)$", table_range)

    if not match:
        raise ValueError(f"Invalid table_range: {table_range}")

    start_col, start_row, end_col, end_row = match.groups()

    return {
        "start_col": column_index_from_string(start_col) - 1,
        "end_col": column_index_from_string(end_col) - 1,
        "start_row": int(start_row) - 1,
        "end_row": int(end_row) - 1 if end_row else None,
    }


def read_excel_table_range(
    local_path: Path,
    sheet_name: str,
    table_range: str,
) -> pd.DataFrame:
    parsed = parse_excel_range(table_range)

    df_raw = pd.read_excel(
        local_path,
        sheet_name=sheet_name,
        header=None,
        dtype=str,
    )

    row_slice = slice(
        parsed["start_row"],
        parsed["end_row"] + 1 if parsed["end_row"] is not None else None,
    )

    col_slice = slice(
        parsed["start_col"],
        parsed["end_col"] + 1,
    )

    df = df_raw.iloc[row_slice, col_slice].copy()

    # Dòng đầu tiên trong range là header
    header = [
        normalize_column_name(col)
        for col in df.iloc[0].tolist()
    ]

    df = df.iloc[1:].reset_index(drop=True)
    df.columns = header

    df = df.dropna(how="all")

    # Bỏ các cột header rỗng / nan
    valid_cols = [
        col for col in df.columns
        if col and col.lower() != "nan"
    ]

    df = df[valid_cols]

    return df


## 4. Test exact object download

In [5]:
def test_download_unique_excels() -> dict[str, Path]:
    local_paths: dict[str, Path] = {}

    unique_sources = sorted({
        config.get("object_key_pattern") or config.get("object_key")
        for config in SHEET_MAPPING.values()
        if config.get("object_key_pattern") or config.get("object_key")
    })

    for source in unique_sources:
        print(f"Resolving/downloading source: s3://{get_hrm_bucket()}/{source}")

        local_path, resolved_object_key, snapshot_date = download_hrm_object(
            object_key_pattern=source if "*" in source else None,
            object_key=source if "*" not in source else None,
        )

        local_paths[source] = local_path

        print("Resolved object:", resolved_object_key)
        print("Snapshot date:", snapshot_date)
        print("Downloaded:", local_path)
        print("Size bytes:", local_path.stat().st_size)
        print("-" * 100)

    return local_paths


LOCAL_EXCEL_PATHS = test_download_unique_excels()


Resolving/downloading source: s3://hrm-data/L&D/L&D_Master Data_*.xlsx
Resolved object: L&D/L&D_Master Data_16062026.xlsx
Snapshot date: 2026-06-16
Downloaded: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_ingestion_pgnvzpyl\L&D_Master Data_16062026.xlsx
Size bytes: 51678789
----------------------------------------------------------------------------------------------------
Resolving/downloading source: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
Resolved object: L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
Snapshot date: 2026-06-23
Downloaded: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_ingestion_g2_qa53r\[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
Size bytes: 178665
----------------------------------------------------------------------------------------------------


## 5. Load mapping/schema/sql

In [6]:
def load_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_column_mapping(domain: str, resource_name: str) -> dict[str, str]:
    return load_json(MAPPING_DIR / domain / f"{resource_name}.json")


def load_parquet_schema(domain: str, resource_name: str) -> Any:
    return load_json(PARQUET_SCHEMA_DIR / domain / f"{resource_name}.json")


def ensure_create_table_ingestion_columns(sql: str) -> str:
    """
    Make CREATE TABLE SQL compatible with notebook-generated ingestion columns.

    Raw Parquet writes snapshot_date as STRING, so Hive/Trino DDL must expose it as VARCHAR,
    not DATE. Also ensure snapshot_date_ts exists as BIGINT.
    """
    updated = sql

    # If snapshot_date already exists but was declared as DATE/STRING/etc., force VARCHAR.
    updated = re.sub(
        r"(?im)^(\s*snapshot_date\s+)(date|string|varchar)(\s*,?)\s*$",
        r"\1varchar\3",
        updated,
    )

    # If snapshot_date_ts already exists but has another integer-ish type, force BIGINT.
    updated = re.sub(
        r"(?im)^(\s*snapshot_date_ts\s+)(int|integer|bigint|long)(\s*,?)\s*$",
        r"\1bigint\3",
        updated,
    )

    has_snapshot_date = re.search(r"(?im)^\s*snapshot_date\s+", updated) is not None
    has_snapshot_date_ts = re.search(r"(?im)^\s*snapshot_date_ts\s+", updated) is not None

    if not has_snapshot_date and not has_snapshot_date_ts:
        # Preferred placement: after filename.
        updated, count = re.subn(
            r"(?im)^(\s*filename\s+varchar\s*,\s*)$",
            r"\1\n   snapshot_date varchar,\n   snapshot_date_ts bigint,",
            updated,
            count=1,
        )

        if count == 0:
            # Fallback: before crawled_at_ts.
            updated = re.sub(
                r"(?im)^(\s*crawled_at_ts\s+bigint\s*,?\s*)$",
                r"   snapshot_date varchar,\n   snapshot_date_ts bigint,\n\1",
                updated,
                count=1,
            )
    elif has_snapshot_date and not has_snapshot_date_ts:
        # Add snapshot_date_ts right after snapshot_date.
        updated = re.sub(
            r"(?im)^(\s*snapshot_date\s+varchar\s*,\s*)$",
            r"\1\n   snapshot_date_ts bigint,",
            updated,
            count=1,
        )
    elif not has_snapshot_date and has_snapshot_date_ts:
        # Add snapshot_date right before snapshot_date_ts.
        updated = re.sub(
            r"(?im)^(\s*snapshot_date_ts\s+bigint\s*,\s*)$",
            r"   snapshot_date varchar,\n\1",
            updated,
            count=1,
        )

    return updated


def load_create_table_sql(domain: str, resource_name: str) -> str:
    path = HIVE_SQL_DIR / domain / f"create_table_{resource_name}.sql"

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    sql = path.read_text(encoding="utf-8")
    return ensure_create_table_ingestion_columns(sql)


print("Mapping exists:", (MAPPING_DIR / "hr_raw" / "hr_training_fresher_list.json").exists())
print("Schema exists:", (PARQUET_SCHEMA_DIR / "hr_raw" / "hr_training_fresher_list.json").exists())
print("SQL exists:", (HIVE_SQL_DIR / "hr_raw" / "create_table_hr_training_fresher_list.sql").exists())


Mapping exists: True
Schema exists: True
SQL exists: True


## 6. Normalize helpers

In [7]:
def normalize_column_name(col: Any) -> str:
    return re.sub(r"\s+", " ", str(col).strip())


def clean_raw_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_column_name(col) for col in df.columns]
    return df


def normalize_schema(schema: Any) -> dict[str, str]:
    """
    Supports:
    1. Spark-style struct:
       {
         "type": "struct",
         "fields": [
           {"name": "...", "dataType": {"type": "string"}}
         ]
       }

    2. Simple dict:
       {"col": "string"}

    3. List:
       [{"name": "col", "type": "string"}]
    """
    if isinstance(schema, dict) and schema.get("type") == "struct":
        fields = schema.get("fields", [])

        return {
            field["name"]: (
                field.get("dataType")
                or field.get("datatype")
            )["type"].lower()
            for field in fields
        }

    if isinstance(schema, dict):
        return {str(k): str(v).lower() for k, v in schema.items()}

    if isinstance(schema, list):
        return {
            str(item["name"]): str(item["type"]).lower()
            for item in schema
        }

    raise ValueError("Unsupported parquet schema format")


def apply_column_mapping(
    df: pd.DataFrame,
    column_mapping: dict[str, str],
    sheet_name: str,
) -> pd.DataFrame:
    df = clean_raw_columns(df)

    normalized_mapping = {
        normalize_column_name(source_col): target_col
        for source_col, target_col in column_mapping.items()
    }

    source_columns = list(normalized_mapping.keys())

    missing_columns = [
        col for col in source_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in sheet '{sheet_name}': {missing_columns}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[source_columns]
    df = df.rename(columns=normalized_mapping)

    return df


def add_metadata_columns(df: pd.DataFrame, object_key: str) -> pd.DataFrame:
    df = df.copy()
    df["filename"] = Path(object_key).name
    df["crawled_at_ts"] = int(time.time())
    return df


def add_timestamp_columns(df: pd.DataFrame, date_columns: list[str]) -> pd.DataFrame:
    df = df.copy()

    for col in date_columns:
        if col not in df.columns:
            continue

        # format="mixed" giúp tránh warning khi dữ liệu đã là YYYY-MM-DD HH:MM:SS
        try:
            parsed = pd.to_datetime(df[col], errors="coerce", format="mixed")
        except TypeError:
            parsed = pd.to_datetime(df[col], errors="coerce")

        ts_col = f"{col}_ts"

        df[ts_col] = parsed.astype("int64") // 10**9
        df.loc[parsed.isna(), ts_col] = pd.NA
        df[ts_col] = df[ts_col].astype("Int64")

    return df


## 7. Apply Parquet schema

In [8]:
PANDAS_TYPE_MAPPING = {
    "string": "string",
    "str": "string",
    "int": "Int64",
    "integer": "Int64",
    "bigint": "Int64",
    "long": "Int64",
    "int64": "Int64",
    "float": "float64",
    "double": "float64",
    "boolean": "boolean",
    "bool": "boolean",
    "date": "string",
    "timestamp": "string",
}


def ensure_ingestion_schema_columns(schema_dict: dict[str, str], df: pd.DataFrame | None = None) -> dict[str, str]:
    """
    Ensure notebook-generated metadata columns are kept even if JSON schema
    has not been updated yet.

    Recommended: still add snapshot_date and snapshot_date_ts to parquet_schema JSON and Hive DDL
    so the table can expose these columns directly.
    """
    schema_dict = dict(schema_dict)

    required_ingestion_cols = {
        "snapshot_date": "string",   # Parquet STRING -> Hive/Trino VARCHAR
        "snapshot_date_ts": "bigint", # Unix timestamp seconds
    }

    # Force correct logical types if columns already exist in JSON with wrong type.
    if "snapshot_date" in schema_dict:
        schema_dict["snapshot_date"] = "string"
    if "snapshot_date_ts" in schema_dict:
        schema_dict["snapshot_date_ts"] = "bigint"

    missing_cols = [col for col in required_ingestion_cols if col not in schema_dict]

    if missing_cols:
        new_schema: dict[str, str] = {}
        inserted = False

        for col, typ in schema_dict.items():
            new_schema[col] = typ

            # Preferred order: filename, snapshot_date, snapshot_date_ts, crawled_at_ts
            if col == "filename":
                for required_col, required_type in required_ingestion_cols.items():
                    if required_col not in schema_dict:
                        new_schema[required_col] = required_type
                inserted = True

        if not inserted:
            for required_col, required_type in required_ingestion_cols.items():
                if required_col not in new_schema:
                    new_schema[required_col] = required_type

        schema_dict = new_schema

    return schema_dict


def apply_parquet_schema(df: pd.DataFrame, schema: Any) -> pd.DataFrame:
    df = df.copy()
    schema_dict = ensure_ingestion_schema_columns(normalize_schema(schema), df)

    for col, logical_type in schema_dict.items():
        if col not in df.columns:
            df[col] = pd.NA

        pandas_type = PANDAS_TYPE_MAPPING.get(logical_type, "string")

        if pandas_type == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif pandas_type == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        elif pandas_type == "boolean":
            df[col] = df[col].astype("boolean")
        else:
            df[col] = df[col].astype("string")

    return df[list(schema_dict.keys())]


schema = load_parquet_schema("hr_raw", "hr_training_fresher_list")
schema_dict = ensure_ingestion_schema_columns(normalize_schema(schema))

print("Schema columns:", len(schema_dict))
print(schema_dict)


Schema columns: 20
{'sequence_number': 'string', 'full_name': 'string', 'employee_code': 'string', 'onboard_date': 'string', 'training_phase': 'string', 'training_major': 'string', 'mentor_name': 'string', 'department_n_2': 'string', 'average_score': 'string', 'ojt_start_date': 'string', 'ojt_end_date': 'string', 'employment_type': 'string', 'training_result': 'string', 'filename': 'string', 'crawled_at_ts': 'long', 'onboard_date_ts': 'long', 'ojt_start_date_ts': 'long', 'ojt_end_date_ts': 'long', 'snapshot_date': 'string', 'snapshot_date_ts': 'bigint'}


## 8. Structure test

In [9]:
def test_excel_structure(resource_name: str, config: dict[str, Any]) -> pd.DataFrame:
    domain = config.get("domain", DEFAULT_DOMAIN)
    source = config.get("object_key_pattern") or config.get("object_key")
    sheet_name = config["sheet_name"]
    header_row = config.get("header_row", 1)
    table_range = config.get("table_range")

    print(f"[{resource_name}] Testing structure")
    print(f"Input source: s3://{get_hrm_bucket()}/{source}")
    print(f"Expected sheet: {sheet_name}")

    if table_range:
        print(f"Table range: {table_range}")
    else:
        print(f"Header row index: {header_row} (Excel row {header_row + 1})")

    local_path, resolved_object_key, snapshot_date = download_hrm_object(
        object_key_pattern=config.get("object_key_pattern"),
        object_key=config.get("object_key"),
    )

    print(f"Resolved object: s3://{get_hrm_bucket()}/{resolved_object_key}")
    print(f"Snapshot date: {snapshot_date}")

    excel = pd.ExcelFile(local_path)

    if sheet_name not in excel.sheet_names:
        print("\nAvailable sheets:")
        for s in excel.sheet_names:
            print(f"- {s}")
        raise ValueError(f"Sheet not found: {sheet_name}")

    if table_range:
        df_preview = read_excel_table_range(local_path, sheet_name, table_range).head(5)
    else:
        df_preview = pd.read_excel(local_path, sheet_name=sheet_name, header=header_row, dtype=str, nrows=5)
        df_preview = clean_raw_columns(df_preview)

    mapping = load_column_mapping(domain, resource_name)
    schema = ensure_ingestion_schema_columns(normalize_schema(load_parquet_schema(domain, resource_name)))

    excel_columns = list(df_preview.columns)
    mapping_source_columns = [normalize_column_name(c) for c in mapping.keys()]
    mapping_target_columns = list(mapping.values())
    schema_columns = list(schema.keys())

    missing_in_excel = [c for c in mapping_source_columns if c not in excel_columns]
    extra_in_excel = [c for c in excel_columns if c not in mapping_source_columns]
    missing_in_schema = [c for c in mapping_target_columns if c not in schema_columns]

    required_extra_cols = ["filename", "snapshot_date", "snapshot_date_ts", "crawled_at_ts"] + [f"{col}_ts" for col in config.get("date_columns", [])]
    missing_extra_cols = [c for c in required_extra_cols if c not in schema_columns]

    print("\nExcel/range columns:")
    print(excel_columns)
    print("\nMapping source columns:")
    print(mapping_source_columns)
    print("\nMapping target columns:")
    print(mapping_target_columns)
    print("\nParquet schema columns:")
    print(schema_columns)
    print("\nMissing mapping source columns in Excel/range:")
    print(missing_in_excel)
    print("\nExtra columns in Excel/range not used by mapping:")
    print(extra_in_excel)
    print("\nMapped target columns missing in Parquet schema:")
    print(missing_in_schema)
    print("\nRequired metadata/timestamp columns missing in effective parquet schema:")
    print(missing_extra_cols)

    if missing_in_excel or missing_in_schema or missing_extra_cols:
        raise ValueError(
            f"[{resource_name}] Structure test failed. "
            "Please fix Excel header/range, mapping JSON, or parquet schema JSON."
        )

    print("\nStructure test passed")
    display(df_preview.head())

    return df_preview


STRUCTURE_TEST_RESULTS: dict[str, pd.DataFrame] = {}
STRUCTURE_TEST_FAILED: dict[str, str] = {}

for resource_name, config in SHEET_MAPPING.items():
    try:
        STRUCTURE_TEST_RESULTS[resource_name] = test_excel_structure(
            resource_name,
            config,
        )
    except Exception as exc:
        STRUCTURE_TEST_FAILED[resource_name] = f"{type(exc).__name__}: {exc}"
        print(f"\n[{resource_name}] STRUCTURE TEST FAILED")
        print(STRUCTURE_TEST_FAILED[resource_name])
        print("=" * 100)

print("\nSUMMARY")
print("Passed:", len(STRUCTURE_TEST_RESULTS))
print("Failed:", len(STRUCTURE_TEST_FAILED))

if STRUCTURE_TEST_FAILED:
    for resource_name, error in STRUCTURE_TEST_FAILED.items():
        print(f"- {resource_name}: {error}")

    raise RuntimeError(
        f"Structure test failed for {len(STRUCTURE_TEST_FAILED)} resources"
    )


[hr_training_class_list] Testing structure
Input source: s3://hrm-data/L&D/L&D_Master Data_*.xlsx
Expected sheet: 01. DS Lớp học
Header row index: 3 (Excel row 4)
Resolved object: s3://hrm-data/L&D/L&D_Master Data_16062026.xlsx
Snapshot date: 2026-06-16

Excel/range columns:
['Unnamed: 0', 'STT', 'Ngày cập nhật', 'Đơn vị tổ chức', 'PIC', 'Mã lớp học', 'Tên lớp học', 'Tên khóa đào tạo', 'Tên chương trình đào tạo', 'Phương pháp', 'Nhóm chương trình', 'Hình thức', 'Trạng thái', 'Thời lượng', 'Ngày bắt đầu', 'Ngày kết thúc', 'Tháng', 'Năm', 'Số lượng kế hoạch', 'Số lượng thực tế', 'Chi phí kế hoạch', 'Chi phí tổ chức', 'Chi phí giảng viên', 'Chi phí mua/thuê ngoài', 'Tổng Thực tế', 'Giảng viên', 'Tỷ lệ tham gia', 'Số giờ học tập', 'Số giờ giảng dạy', 'Số giờ đào tạo', 'Số lượng', 'Tỷ lệ tham gia đánh giá', 'Nội dung', 'Giảng viên.1', 'Công tác tổ chức', 'Trung bình', 'Đánh giá cấp độ 2', 'Tỷ lệ học viên "Đạt"', 'PIC.1', 'Ghi chú trạng thái LXP']

Mapping source columns:
['STT', 'Ngày cập n

,Unnamed: 0,STT,Ngày cập nhật,Đơn vị tổ chức,PIC,Mã lớp học,Tên lớp học,Tên khóa đào tạo,Tên chương trình đào tạo,Phương pháp,...,Số lượng,Tỷ lệ tham gia đánh giá,Nội dung,Giảng viên.1,Công tác tổ chức,Trung bình,Đánh giá cấp độ 2,"Tỷ lệ học viên ""Đạt""",PIC.1,Ghi chú trạng thái LXP
0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0,06/25,VCS,AnhNTL11,PTSPDT_25.01,Nâng cao Năng lực Phát triển Sản phẩm với Desi...,Design Thinking,Design Thinking,Kết hợp,...,NaN,0,4.5,4.5,4.5,4.5,NaN,0,AnhNTL11,NaN
2,NaN,0,06/25,VCS,AnhNTL11,WKD_25.01,Workshop quý I_Khối SPDV & Khối Kinh doanh_ ph...,Workshop Kinh doanh quý I,Workshop Kinh doanh quý,Kết hợp,...,NaN,0,NaN,NaN,NaN,4.665,NaN,0,AnhNTL11,NaN
3,NaN,0,06/25,VCS,AnhNTL11,WKD_25.02,Workshop quý I_Khối SPDV & Khối Kinh doanh_ ph...,Workshop Kinh doanh quý I,Workshop Kinh doanh quý,Kết hợp,...,NaN,0,NaN,NaN,NaN,4.665,NaN,0,AnhNTL11,NaN
4,NaN,0,06/25,VCS,NgaDT,TQSPDV _25.01,Tổng quan SPDV,AM_JobOnboarding_T4.01,Đào tạo Job Onboarding cho AM,Trực tiếp,...,NaN,0,5,5,5,5,Yes,1,NgaDT,NaN


[hr_training_instructor_list] Testing structure
Input source: s3://hrm-data/L&D/L&D_Master Data_*.xlsx
Expected sheet: 02. DS Giảng viên
Header row index: 2 (Excel row 3)
Resolved object: s3://hrm-data/L&D/L&D_Master Data_16062026.xlsx
Snapshot date: 2026-06-16

Excel/range columns:
['Unnamed: 0', 'Mã nhân viên', 'Họ và tên', 'Email', 'Nhóm giảng viên', 'Khối', 'Phòng ban', 'Mã lớp học', 'Tên lớp học', 'Thời lượng giảng dạy', 'Chi phí hỗ trợ', 'Đánh giá giảng viên', 'Ngày bắt đầu', 'Ngày kết thúc', 'Tháng', 'Năm']

Mapping source columns:
['Mã nhân viên', 'Họ và tên', 'Email', 'Nhóm giảng viên', 'Khối', 'Phòng ban', 'Mã lớp học', 'Tên lớp học', 'Thời lượng giảng dạy', 'Chi phí hỗ trợ', 'Đánh giá giảng viên', 'Ngày bắt đầu', 'Ngày kết thúc', 'Tháng', 'Năm']

Mapping target columns:
['employee_code', 'full_name', 'business_email', 'instructor_group', 'division_n', 'department_n_2', 'class_code', 'class_name', 'teaching_duration', 'instructor_fee', 'instructor_rating', 'start_date', 'end_

,Unnamed: 0,Mã nhân viên,Họ và tên,Email,Nhóm giảng viên,Khối,Phòng ban,Mã lớp học,Tên lớp học,Thời lượng giảng dạy,Chi phí hỗ trợ,Đánh giá giảng viên,Ngày bắt đầu,Ngày kết thúc,Tháng,Năm
0,NaN,xxxxxx,Nguyễn A,NaN,4,NaN,NaN,LTATTT-25.01,NaN,12,2400000,NaN,NaN,NaN,NaN,NaN
1,NaN,266817,Nguyễn Khánh Hưng,hungnk16@viettel.com.vn,4,Khối Cơ quan,Phòng Trải nghiệm khách hàng,PTSPDT_25.01,Nâng cao Năng lực Phát triển Sản phẩm với Desi...,16,0,NaN,2025-01-14 00:00:00,2025-01-15 00:00:00,1,2025
2,NaN,NaN,Nguyễn Anh Tuấn,tuanna2@viettel.com,2,Khối Lãnh đạo,Ban Giám đốc,PTBSPDV_25.01,Phương thức bán SPDV VCS,NaN,NaN,NaN,2025-04-14 00:00:00,2025-04-14 00:00:00,4,2025
3,NaN,NaN,Nguyễn Văn Huy,huynv20@viettel.com,3,Khối Dịch vụ,TT. Giám sát và phản ứng trên KGM,TTKHĐT_25.01,"Thị trường, khách hàng, đối thủ",NaN,NaN,NaN,2025-04-11 00:00:00,2025-04-11 00:00:00,4,2025
4,NaN,NaN,Hoàng Thu Hà,haht31@viettel.com,5,Khối Kinh doanh,Phòng Chiến lược Kinh doanh,QTKD_25.01,Quy trình Kinh doanh,NaN,NaN,NaN,2025-04-11 00:00:00,2025-04-11 00:00:00,4,2025


[hr_training_learner_list] Testing structure
Input source: s3://hrm-data/L&D/L&D_Master Data_*.xlsx
Expected sheet: 03. DS Học viên
Header row index: 2 (Excel row 3)
Resolved object: s3://hrm-data/L&D/L&D_Master Data_16062026.xlsx
Snapshot date: 2026-06-16

Excel/range columns:
['Unnamed: 0', 'Mã nhân viên', 'Họ và tên', 'Diện', 'Khối', 'Phòng ban', 'Mã lớp học', 'Tên lớp học', 'Tên khóa học', 'Kết quả', 'Thời lượng (giờ)', 'Ngày bắt đầu', 'Ngày kết thúc', 'Tháng', 'Năm', 'Ghi chú trạng thái LXP']

Mapping source columns:
['Mã nhân viên', 'Họ và tên', 'Diện', 'Khối', 'Phòng ban', 'Mã lớp học', 'Tên lớp học', 'Tên khóa học', 'Kết quả', 'Thời lượng (giờ)', 'Thời lượng (giờ)', 'Ngày bắt đầu', 'Ngày bắt đầu', 'Ngày kết thúc', 'Ngày kết thúc', 'Tháng', 'Năm', 'Ghi chú trạng thái LXP']

Mapping target columns:
['employee_code', 'full_name', 'employee_object', 'division_n', 'department_n_2', 'class_code', 'class_name', 'course_name', 'learning_result', 'learning_duration_hours', 'learning_dur

...

[hr_training_fresher_conversion_list] Testing structure
Input source: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
Expected sheet: DS Fresher chuyển diện
Header row index: 1 (Excel row 2)
Resolved object: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
Snapshot date: 2026-06-23

Excel/range columns:
['STT', 'Họ và tên', 'Mã nhân viên', 'Ngày onboard', 'Ngày chuyển diện', 'Vị trí', 'Level - vùng', 'Phòng ban', 'Thời gian bắt đầu OJT', 'Thời gian kết thúc OJT', 'Văn phòng làm việc hiện tại', 'KQ phỏng vấn', 'File kq', 'Email Viettel', 'Email cá nhân', 'Ngày sinh', 'Số điện thoại', 'Link theo dõi kết quả đào tạo', 'Tiến độ hiện tại', 'Số CCCD', 'Ngày cấp', 'Nơi cấp', 'Địa chỉ thường trú', 'Chỗ ở hiện tại', 'Giới tính', 'Trình độ', 'Trường', 'Chuyên ngành', 'Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA)', 'Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ "Dự kiến") (Ghi rõ tháng/năm)', 'Điểm thi IQ', 'Tiếng Anh (TOEIC, IELTS...) (Gh

,STT,Họ và tên,Mã nhân viên,Ngày onboard,Ngày chuyển diện,Vị trí,Level - vùng,Phòng ban,Thời gian bắt đầu OJT,Thời gian kết thúc OJT,...,Trình độ,Trường,Chuyên ngành,Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA),"Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ ""Dự kiến"") (Ghi rõ tháng/năm)",Điểm thi IQ,"Tiếng Anh (TOEIC, IELTS...) (Ghi rõ loại chứng chỉ + số điểm) (Nếu mới thi E-learning mà chưa có chứng chỉ cũng ghi rõ)",Điểm thi chuyên môn,Số sổ BHXH,Mã số thuế
0,1,Tạ Duy Tân,809251,2023-08-11 00:00:00,2024-01-01 00:00:00,NV GS nguy cơ ATTT,Junior vùng 1,Trung tâm Phân tích chia sẻ nguy cơ An ninh mạng,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Bùi Trọng Hiếu,809252,2023-08-11 00:00:00,2024-01-01 00:00:00,NV GS nguy cơ ATTT,Junior vùng 1,Trung tâm Phân tích chia sẻ nguy cơ An ninh mạng,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Nguyễn Ngọc Huân,809367,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Nguyễn Bình Minh,809369,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Đỗ Văn Hiếu,809364,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


[hr_training_fresher_resignation_list] Testing structure
Input source: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
Expected sheet: DS Fresher nghỉ việc
Header row index: 1 (Excel row 2)
Resolved object: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
Snapshot date: 2026-06-23

Excel/range columns:
['STT', 'Họ và tên', 'Mã nhân viên', 'Ngày onboard', 'Ngày nghỉ việc', 'Lý do', 'Giai đoạn đào tạo', 'Chuyên ngành đào tạo', 'Mentor trực tiếp', 'Phòng ban', 'Văn phòng làm việc hiện tại', 'Unnamed: 11', 'Email Viettel', 'Email cá nhân', 'Ngày sinh', 'Số điện thoại', 'Link theo dõi kết quả đào tạo', 'Tiến độ hiện tại', 'Số CCCD', 'Ngày cấp', 'Nơi cấp', 'Địa chỉ thường trú', 'Chỗ ở hiện tại', 'Giới tính', 'Trình độ', 'Trường', 'Chuyên ngành', 'Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA)', 'Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ "Dự kiến") (Ghi rõ tháng/năm)', 'Điểm thi IQ', 'Tiếng Anh (TOEIC, IELTS...) (Ghi rõ loại chứng ch

,STT,Họ và tên,Mã nhân viên,Ngày onboard,Ngày nghỉ việc,Lý do,Giai đoạn đào tạo,Chuyên ngành đào tạo,Mentor trực tiếp,Phòng ban,...,Trình độ,Trường,Chuyên ngành,Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA),"Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ ""Dự kiến"") (Ghi rõ tháng/năm)",Điểm thi IQ,"Tiếng Anh (TOEIC, IELTS...) (Ghi rõ loại chứng chỉ + số điểm) (Nếu mới thi E-learning mà chưa có chứng chỉ cũng ghi rõ)",Điểm thi chuyên môn,Số sổ BHXH,Mã số thuế
0,1,Nguyễn Đức Thắng,817904,2024-01-18 00:00:00,2024-02-07 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Trịnh Trường Giang,817925,2024-01-18 00:00:00,2024-02-29 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Nguyễn Đức Trường Giang,817908,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Nguyễn Thanh Tùng,817910,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Nguyễn Trần Đức An,817898,2024-01-18 00:00:00,2024-04-15 00:00:00,Xin nghỉ do thay đổi công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



SUMMARY
Passed: 14
Failed: 0


In [10]:
FAILED_RESOURCES = [
    "hr_training_fresher_resignation_list",
]

for resource_name in FAILED_RESOURCES:
    print("\n" + "=" * 120)
    print(resource_name)
    print("=" * 120)

    try:
        test_excel_structure(resource_name, SHEET_MAPPING[resource_name])
    except Exception as exc:
        print(type(exc).__name__, exc)


hr_training_fresher_resignation_list
[hr_training_fresher_resignation_list] Testing structure
Input source: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
Expected sheet: DS Fresher nghỉ việc
Header row index: 1 (Excel row 2)
Resolved object: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
Snapshot date: 2026-06-23

Excel/range columns:
['STT', 'Họ và tên', 'Mã nhân viên', 'Ngày onboard', 'Ngày nghỉ việc', 'Lý do', 'Giai đoạn đào tạo', 'Chuyên ngành đào tạo', 'Mentor trực tiếp', 'Phòng ban', 'Văn phòng làm việc hiện tại', 'Unnamed: 11', 'Email Viettel', 'Email cá nhân', 'Ngày sinh', 'Số điện thoại', 'Link theo dõi kết quả đào tạo', 'Tiến độ hiện tại', 'Số CCCD', 'Ngày cấp', 'Nơi cấp', 'Địa chỉ thường trú', 'Chỗ ở hiện tại', 'Giới tính', 'Trình độ', 'Trường', 'Chuyên ngành', 'Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA)', 'Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ "Dự kiến") (Ghi rõ tháng/năm)', 'Điểm thi IQ', 'Tiếng Anh (

,STT,Họ và tên,Mã nhân viên,Ngày onboard,Ngày nghỉ việc,Lý do,Giai đoạn đào tạo,Chuyên ngành đào tạo,Mentor trực tiếp,Phòng ban,...,Trình độ,Trường,Chuyên ngành,Loại tốt nghiệp (nếu chưa tốt nghiệp thì nhập GPA),"Thời gian tốt nghiệp (nếu chưa tốt nghiệp thì thêm cụm từ ""Dự kiến"") (Ghi rõ tháng/năm)",Điểm thi IQ,"Tiếng Anh (TOEIC, IELTS...) (Ghi rõ loại chứng chỉ + số điểm) (Nếu mới thi E-learning mà chưa có chứng chỉ cũng ghi rõ)",Điểm thi chuyên môn,Số sổ BHXH,Mã số thuế
0,1,Nguyễn Đức Thắng,817904,2024-01-18 00:00:00,2024-02-07 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Trịnh Trường Giang,817925,2024-01-18 00:00:00,2024-02-29 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Nguyễn Đức Trường Giang,817908,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Nguyễn Thanh Tùng,817910,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Nguyễn Trần Đức An,817898,2024-01-18 00:00:00,2024-04-15 00:00:00,Xin nghỉ do thay đổi công việc,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. Read and normalize full data

In [11]:
def read_excel_sheet_resource(resource_name: str, config: dict[str, Any]) -> pd.DataFrame:
    domain = config.get("domain", DEFAULT_DOMAIN)
    source = config.get("object_key_pattern") or config.get("object_key")
    sheet_name = config["sheet_name"]
    header_row = config.get("header_row", 1)
    drop_rows = config.get("drop_rows", 0)
    table_range = config.get("table_range")

    print(f"[{resource_name}] Resolving/downloading s3://{get_hrm_bucket()}/{source}")

    local_path, resolved_object_key, snapshot_date = download_hrm_object(
        object_key_pattern=config.get("object_key_pattern"),
        object_key=config.get("object_key"),
    )

    print(f"[{resource_name}] Resolved object: s3://{get_hrm_bucket()}/{resolved_object_key}")
    print(f"[{resource_name}] Snapshot date: {snapshot_date}")
    print(f"[{resource_name}] Reading sheet: {sheet_name}")

    if table_range:
        print(f"[{resource_name}] Reading table range: {table_range}")
        df = read_excel_table_range(local_path=local_path, sheet_name=sheet_name, table_range=table_range)
    else:
        df = pd.read_excel(local_path, sheet_name=sheet_name, header=header_row, dtype=str)

        if drop_rows > 0:
            df = df.iloc[drop_rows:].reset_index(drop=True)

        df = df.dropna(how="all")

    mapping = load_column_mapping(domain, resource_name)
    schema = load_parquet_schema(domain, resource_name)

    df = apply_column_mapping(df, mapping, sheet_name)
    df = add_metadata_columns(df, resolved_object_key)
    df["snapshot_date"] = snapshot_date.isoformat() if snapshot_date else None
    df["snapshot_date_ts"] = date_to_unix_seconds(snapshot_date)
    df = add_timestamp_columns(df, config.get("date_columns", []))
    df = apply_parquet_schema(df, schema)

    print(f"[{resource_name}] Loaded {len(df)} rows, {len(df.columns)} columns")

    return df


PROCESSED_DATA: dict[str, pd.DataFrame] = {}

for resource_name, config in SHEET_MAPPING.items():
    df = read_excel_sheet_resource(resource_name, config)
    PROCESSED_DATA[resource_name] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


[hr_training_class_list] Resolving/downloading s3://hrm-data/L&D/L&D_Master Data_*.xlsx
[hr_training_class_list] Resolved object: s3://hrm-data/L&D/L&D_Master Data_16062026.xlsx
[hr_training_class_list] Snapshot date: 2026-06-16
[hr_training_class_list] Reading sheet: 01. DS Lớp học
[hr_training_class_list] Loaded 874 rows, 34 columns
Shape: (874, 34)
Columns: ['sequence_number', 'updated_date', 'organizing_unit', 'pic', 'class_code', 'class_name', 'course_name', 'training_program', 'training_method', 'program_group', 'training_format', 'class_status', 'training_duration', 'start_date', 'end_date', 'month', 'year', 'planned_learner_count', 'actual_learner_count', 'training_hours', 'training_content', 'instructor_name', 'organization_rating', 'average_rating', 'level_2_evaluation', 'pass_rate', 'lxp_status_note', 'filename', 'crawled_at_ts', 'updated_date_ts', 'start_date_ts', 'end_date_ts', 'snapshot_date', 'snapshot_date_ts']


c:\Users\giangnth46\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,sequence_number,updated_date,organizing_unit,pic,class_code,class_name,course_name,training_program,training_method,program_group,...,level_2_evaluation,pass_rate,lxp_status_note,filename,crawled_at_ts,updated_date_ts,start_date_ts,end_date_ts,snapshot_date,snapshot_date_ts
0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,L&D_Master Data_16062026.xlsx,1782655592,<NA>,<NA>,<NA>,2026-06-16,1781568000
1,0,06/25,VCS,AnhNTL11,PTSPDT_25.01,Nâng cao Năng lực Phát triển Sản phẩm với Desi...,Design Thinking,Design Thinking,Kết hợp,"Chuyên môn, nghiệp vụ",...,<NA>,0,<NA>,L&D_Master Data_16062026.xlsx,1782655592,<NA>,1736812800,1736899200,2026-06-16,1781568000
2,0,06/25,VCS,AnhNTL11,WKD_25.01,Workshop quý I_Khối SPDV & Khối Kinh doanh_ ph...,Workshop Kinh doanh quý I,Workshop Kinh doanh quý,Kết hợp,"Chuyên môn, nghiệp vụ",...,<NA>,0,<NA>,L&D_Master Data_16062026.xlsx,1782655592,<NA>,1739318400,1739318400,2026-06-16,1781568000
3,0,06/25,VCS,AnhNTL11,WKD_25.02,Workshop quý I_Khối SPDV & Khối Kinh doanh_ ph...,Workshop Kinh doanh quý I,Workshop Kinh doanh quý,Kết hợp,"Chuyên môn, nghiệp vụ",...,<NA>,0,<NA>,L&D_Master Data_16062026.xlsx,1782655592,<NA>,1739491200,1739491200,2026-06-16,1781568000
4,0,06/25,VCS,NgaDT,TQSPDV _25.01,Tổng quan SPDV,AM_JobOnboarding_T4.01,Đào tạo Job Onboarding cho AM,Trực tiếp,Nhân viên mới,...,Yes,1,<NA>,L&D_Master Data_16062026.xlsx,1782655592,<NA>,1744070400,1744070400,2026-06-16,1781568000


[hr_training_instructor_list] Resolving/downloading s3://hrm-data/L&D/L&D_Master Data_*.xlsx
[hr_training_instructor_list] Resolved object: s3://hrm-data/L&D/L&D_Master Data_16062026.xlsx
[hr_training_instructor_list] Snapshot date: 2026-06-16
[hr_training_instructor_list] Reading sheet: 02. DS Giảng viên
[hr_training_instructor_list] Loaded 151 rows, 21 columns
Shape: (151, 21)
Columns: ['employee_code', 'full_name', 'business_email', 'instructor_group', 'division_n', 'department_n_2', 'class_code', 'class_name', 'teaching_duration', 'instructor_fee', 'instructor_rating', 'start_date', 'end_date', 'month', 'year', 'filename', 'crawled_at_ts', 'start_date_ts', 'end_date_ts', 'snapshot_date', 'snapshot_date_ts']


c:\Users\giangnth46\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


...

,sequence_number,full_name,employee_code,onboard_date,training_phase,training_major,mentor_name,department_n_2,average_score,ojt_start_date,ojt_end_date,employment_type,training_result,filename,crawled_at_ts,onboard_date_ts,ojt_start_date_ts,ojt_end_date_ts,snapshot_date,snapshot_date_ts
0,1,Đàm Thanh Bách,480242,2025-08-12 00:00:00,OJT,Tier 3,Phạm Minh Quân,SOC,8,2026-02-27 00:00:00,2026-05-27 00:00:00,Parttime.,https://docs.google.com/spreadsheets/d/1mTtlA3...,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655683,1754956800,1772150400,1779840000,2026-06-23,1782172800
1,2,Nguyễn Đức Long,480628,2025-08-25 00:00:00,OJT,Tier 3,Nguyễn Anh Quân,SOC,"7,6",2026-02-27 00:00:00,2026-05-27 00:00:00,Fulltime.,https://docs.google.com/spreadsheets/d/1EP7UJ8...,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655683,1756080000,1772150400,1779840000,2026-06-23,1782172800
2,3,Võ Anh Khôi,852408,2025-08-25 00:00:00,OJT,Tier 3,Đỗ Đức Minh,SOC,"7,5",2026-02-27 00:00:00,2026-05-27 00:00:00,Fulltime.,https://docs.google.com/spreadsheets/d/1tlSCtP...,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655683,1756080000,1772150400,1779840000,2026-06-23,1782172800
3,4,Nguyễn Tiến Thành,481860,2025-09-29 00:00:00,OJT,Tier 3,Nguyễn Tùng Anh,SOC,"7,3",2026-02-27 00:00:00,2026-05-27 00:00:00,Fulltime.,https://docs.google.com/spreadsheets/d/1v1GPCw...,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655683,1759104000,1772150400,1779840000,2026-06-23,1782172800
4,5,Lê Thị Ngọc Thảo,854094,2025-09-29 00:00:00,OJT,Tier 3,Tạ Minh Quân,SOC,"7,7",2026-02-27 00:00:00,2026-05-27 00:00:00,Fulltime,https://docs.google.com/spreadsheets/d/1VEw-aw...,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655683,1759104000,1772150400,1779840000,2026-06-23,1782172800


[hr_training_fresher_conversion_list] Resolving/downloading s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
[hr_training_fresher_conversion_list] Resolved object: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
[hr_training_fresher_conversion_list] Snapshot date: 2026-06-23
[hr_training_fresher_conversion_list] Reading sheet: DS Fresher chuyển diện
[hr_training_fresher_conversion_list] Loaded 80 rows, 17 columns
Shape: (80, 17)
Columns: ['sequence_number', 'full_name', 'employee_code', 'onboard_date', 'conversion_date', 'role_base', 'level_region', 'department_n_2', 'current_work_location', 'interview_result', 'interview_result_file', 'filename', 'crawled_at_ts', 'onboard_date_ts', 'conversion_date_ts', 'snapshot_date', 'snapshot_date_ts']


,sequence_number,full_name,employee_code,onboard_date,conversion_date,role_base,level_region,department_n_2,current_work_location,interview_result,interview_result_file,filename,crawled_at_ts,onboard_date_ts,conversion_date_ts,snapshot_date,snapshot_date_ts
0,1,Tạ Duy Tân,809251,2023-08-11 00:00:00,2024-01-01 00:00:00,NV GS nguy cơ ATTT,Junior vùng 1,Trung tâm Phân tích chia sẻ nguy cơ An ninh mạng,<NA>,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1691712000,1704067200,2026-06-23,1782172800
1,2,Bùi Trọng Hiếu,809252,2023-08-11 00:00:00,2024-01-01 00:00:00,NV GS nguy cơ ATTT,Junior vùng 1,Trung tâm Phân tích chia sẻ nguy cơ An ninh mạng,<NA>,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1691712000,1704067200,2026-06-23,1782172800
2,3,Nguyễn Ngọc Huân,809367,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,<NA>,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1692057600,1704067200,2026-06-23,1782172800
3,4,Nguyễn Bình Minh,809369,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,<NA>,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1692057600,1704067200,2026-06-23,1782172800
4,5,Đỗ Văn Hiếu,809364,2023-08-15 00:00:00,2024-01-01 00:00:00,NV Giám sát ATTT Tier 1,Junior vùng 1,Trung tâm GS&PU trên KGM,<NA>,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1692057600,1704067200,2026-06-23,1782172800


[hr_training_fresher_resignation_list] Resolving/downloading s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_*.xlsx
[hr_training_fresher_resignation_list] Resolved object: s3://hrm-data/L&D/[VCS] Quản lý chung sinh viên & Fresher_23062026.xlsx
[hr_training_fresher_resignation_list] Snapshot date: 2026-06-23
[hr_training_fresher_resignation_list] Reading sheet: DS Fresher nghỉ việc
[hr_training_fresher_resignation_list] Loaded 38 rows, 21 columns
Shape: (38, 21)
Columns: ['sequence_number', 'full_name', 'employee_code', 'onboard_date', 'resigned_date', 'resignation_reason', 'training_phase', 'training_major', 'mentor_name', 'department_n_2', 'ojt_start_date', 'ojt_end_date', 'current_work_location', 'filename', 'crawled_at_ts', 'onboard_date_ts', 'resigned_date_ts', 'ojt_start_date_ts', 'ojt_end_date_ts', 'snapshot_date', 'snapshot_date_ts']


,sequence_number,full_name,employee_code,onboard_date,resigned_date,resignation_reason,training_phase,training_major,mentor_name,department_n_2,...,ojt_end_date,current_work_location,filename,crawled_at_ts,onboard_date_ts,resigned_date_ts,ojt_start_date_ts,ojt_end_date_ts,snapshot_date,snapshot_date_ts
0,1,Nguyễn Đức Thắng,817904,2024-01-18 00:00:00,2024-02-07 00:00:00,Không đáp ứng yêu cầu công việc,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1705536000,1707264000,<NA>,<NA>,2026-06-23,1782172800
1,2,Trịnh Trường Giang,817925,2024-01-18 00:00:00,2024-02-29 00:00:00,Không đáp ứng yêu cầu công việc,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1705536000,1709164800,<NA>,<NA>,2026-06-23,1782172800
2,3,Nguyễn Đức Trường Giang,817908,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1705536000,1711497600,<NA>,<NA>,2026-06-23,1782172800
3,4,Nguyễn Thanh Tùng,817910,2024-01-18 00:00:00,2024-03-27 00:00:00,Không đáp ứng yêu cầu công việc,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1705536000,1711497600,<NA>,<NA>,2026-06-23,1782172800
4,5,Nguyễn Trần Đức An,817898,2024-01-18 00:00:00,2024-04-15 00:00:00,Xin nghỉ do thay đổi công việc,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,[VCS] Quản lý chung sinh viên & Fresher_230620...,1782655684,1705536000,1713139200,<NA>,<NA>,2026-06-23,1782172800


## 10. Write Parquet local

In [12]:
def write_parquet_local(df: pd.DataFrame, resource_name: str) -> Path:
    output_dir = Path(tempfile.mkdtemp(prefix="hrm_parquet_"))
    output_path = output_dir / f"{resource_name}.parquet"

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, output_path)

    return output_path


LOCAL_PARQUET_FILES: dict[str, Path] = {}

for resource_name, df in PROCESSED_DATA.items():
    parquet_path = write_parquet_local(df, resource_name)
    LOCAL_PARQUET_FILES[resource_name] = parquet_path

    print(f"[{resource_name}] Local parquet:", parquet_path)
    print(f"[{resource_name}] Size bytes:", parquet_path.stat().st_size)


[hr_training_class_list] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_parquet_3fggueld\hr_training_class_list.parquet
[hr_training_class_list] Size bytes: 37316
[hr_training_instructor_list] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_parquet_pxe38thb\hr_training_instructor_list.parquet
[hr_training_instructor_list] Size bytes: 21712
[hr_training_learner_list] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_parquet_1np1cv9p\hr_training_learner_list.parquet
[hr_training_learner_list] Size bytes: 53575
[hr_training_online_learning_list] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_parquet__zz6cc3p\hr_training_online_learning_list.parquet
[hr_training_online_learning_list] Size bytes: 21909
[hr_training_employee_certificate_list] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_parquet_g8kj2asp\hr_training_employee_certificate_list.parquet
[hr_training_employee_certificate_list] Size bytes: 32758
[hr_training_budget_summary_by_type] Local p

## 11. Upload Parquet to raw bucket

In [13]:
def domain_to_s3_prefix(domain: str) -> str:
    return domain.replace("_", "-")


def delete_raw_resource_prefix(
    client: Minio,
    bucket: str,
    domain: str,
    resource_name: str,
) -> None:
    """Delete old parquet files for this resource before uploading new output."""
    domain_dash = domain_to_s3_prefix(domain)
    prefix = f"{domain_dash}/{resource_name}/"

    objects = list(client.list_objects(bucket, prefix=prefix, recursive=True))

    if not objects:
        print(f"[{resource_name}] No old raw objects to delete under s3a://{bucket}/{prefix}")
        return

    for obj in objects:
        client.remove_object(bucket, obj.object_name)

    print(f"[{resource_name}] Deleted {len(objects)} old raw objects under s3a://{bucket}/{prefix}")


def upload_parquet_to_raw(
    parquet_path: Path,
    domain: str,
    resource_name: str,
    delete_old: bool = True,
) -> str:
    bucket = get_raw_bucket()
    client = get_raw_client()

    domain_dash = domain_to_s3_prefix(domain)
    object_name = f"{domain_dash}/{resource_name}/{parquet_path.name}"

    if delete_old:
        delete_raw_resource_prefix(
            client=client,
            bucket=bucket,
            domain=domain,
            resource_name=resource_name,
        )

    client.fput_object(
        bucket_name=bucket,
        object_name=object_name,
        file_path=str(parquet_path),
        content_type="application/octet-stream",
    )

    return f"s3a://{bucket}/{domain_dash}/{resource_name}"


UPLOAD_LOCATIONS: dict[str, str] = {}

for resource_name, parquet_path in LOCAL_PARQUET_FILES.items():
    config = SHEET_MAPPING[resource_name]
    domain = config.get("domain", DEFAULT_DOMAIN)

    location = upload_parquet_to_raw(
        parquet_path=parquet_path,
        domain=domain,
        resource_name=resource_name,
        delete_old=True,
    )

    UPLOAD_LOCATIONS[resource_name] = location
    print(f"[{resource_name}] Uploaded to {location}")


[hr_training_class_list] Deleted 1 old raw objects under s3a://vcs-raw/hr-raw/hr_training_class_list/
[hr_training_class_list] Uploaded to s3a://vcs-raw/hr-raw/hr_training_class_list
[hr_training_instructor_list] Deleted 1 old raw objects under s3a://vcs-raw/hr-raw/hr_training_instructor_list/
[hr_training_instructor_list] Uploaded to s3a://vcs-raw/hr-raw/hr_training_instructor_list
[hr_training_learner_list] Deleted 1 old raw objects under s3a://vcs-raw/hr-raw/hr_training_learner_list/
[hr_training_learner_list] Uploaded to s3a://vcs-raw/hr-raw/hr_training_learner_list
[hr_training_online_learning_list] Deleted 1 old raw objects under s3a://vcs-raw/hr-raw/hr_training_online_learning_list/
[hr_training_online_learning_list] Uploaded to s3a://vcs-raw/hr-raw/hr_training_online_learning_list
[hr_training_employee_certificate_list] Deleted 1 old raw objects under s3a://vcs-raw/hr-raw/hr_training_employee_certificate_list/
[hr_training_employee_certificate_list] Uploaded to s3a://vcs-raw/hr

## 12. Execute SQL create table - final step

In [15]:
def execute_sql(sql: str) -> None:
    dry_run = os.getenv("DRY_RUN_SQL", "true").lower() == "false"

    if dry_run:
        print("DRY_RUN_SQL=true, skip execute SQL")
        print(sql)
        return

    import trino

    host = os.getenv("TRINO_HOST")
    user = os.getenv("TRINO_USERNAME")

    if not host or not user:
        raise ValueError(
            "Missing TRINO_HOST or TRINO_USER. "
            "Set DRY_RUN_SQL=true if you only want to print SQL."
        )

    conn = trino.dbapi.connect(
        host=host,
        port=int(os.getenv("TRINO_PORT", "443")),
        user=user,
        catalog=os.getenv("TRINO_CATALOG", "hive"),
        schema=os.getenv("TRINO_SCHEMA", DEFAULT_DOMAIN),
        http_scheme=os.getenv("TRINO_HTTP_SCHEME", "https"),
    )

    cur = conn.cursor()
    cur.execute(sql)
    cur.fetchall()
    cur.close()
    conn.close()


for resource_name, config in SHEET_MAPPING.items():
    domain = config.get("domain", DEFAULT_DOMAIN)
    sql = load_create_table_sql(domain, resource_name)

    print(f"[{resource_name}] Execute create table SQL")
    execute_sql(sql)

print(f"All done. Executed SQL step for {len(SHEET_MAPPING)} resources.")


[hr_training_class_list] Execute create table SQL
DRY_RUN_SQL=true, skip execute SQL
CREATE TABLE IF NOT EXISTS hr_raw.hr_training_class_list (
    sequence_number VARCHAR,
    updated_date VARCHAR,
    organizing_unit VARCHAR,
    pic VARCHAR,
    class_code VARCHAR,
    class_name VARCHAR,
    course_name VARCHAR,
    training_program VARCHAR,
    training_method VARCHAR,
    program_group VARCHAR,
    training_format VARCHAR,
    class_status VARCHAR,
    training_duration VARCHAR,
    start_date VARCHAR,
    end_date VARCHAR,
    month VARCHAR,
    year VARCHAR,
    planned_learner_count VARCHAR,
    actual_learner_count VARCHAR,
    training_hours VARCHAR,
    training_content VARCHAR,
    instructor_name VARCHAR,
    organization_rating VARCHAR,
    average_rating VARCHAR,
    level_2_evaluation VARCHAR,
    pass_rate VARCHAR,
    lxp_status_note VARCHAR,
    updated_date_ts BIGINT,
    start_date_ts BIGINT,
    end_date_ts BIGINT,
    snapshot_date varchar,
    snapshot_date_ts 

In [ ]:
for resource_name, location in UPLOAD_LOCATIONS.items():
    print(resource_name)
    print(location)

In [ ]:
df = PROCESSED_DATA["hr_training_fresher_list"]

print(df.shape)
display(df.head())